In [1]:
import pandas as pd
import numpy as np


from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import RandomizedSearchCV



In [2]:
pinch_hitter_record = pd.read_csv("06.pinch_hitter_record_final.csv")
pinch_hitter_record['team'].unique()

array(['樂天桃猿', '味全龍', '台鋼雄鷹', '富邦悍將', '統一7-ELEVEn獅', '中信兄弟'], dtype=object)

In [3]:
pinch_hitter_record.head()

,team,pinch_hitter,pinch_hitter_hand,pinch_hitter_ops,pinch_hitter_history_ops,pinch_hitter_ops_with_pitcherHand,original_batter,original_batter_hand,original_batter_ops,original_batter_history_ops,original_batter_ops_with_pitcherHand,pitcher_hand,platoon_effect,ops_diff,ops_history_diff,ops_with_pitcherHand_diff,cluster
0,樂天桃猿,林泓育,-1.033441,0.253277,0.353504,0.110873,嚴宏鈞,1.134729,-1.256855,-0.577277,-0.419515,-0.567671,-0.853872,0.743110,0.643414,0.431531,1
1,味全龍,鄭鎧文,-1.033441,-0.271839,-1.328013,0.152632,張祐銘,1.134729,-0.398269,-0.730176,0.574805,1.761583,1.171136,-0.239916,-0.363619,-0.273266,0
2,台鋼雄鷹,黃劼希,-1.033441,0.269927,0.119293,0.183870,林家鋐,1.134729,-2.687689,-0.231840,-2.156622,-0.567671,-0.853872,1.976402,0.245377,1.483380,1
3,樂天桃猿,林泓育,-1.033441,-2.492937,0.353504,-2.073853,蔡鎮宇,-0.881267,-2.687689,-3.652236,-0.721286,-0.567671,-0.853872,-0.029820,2.804754,-0.770188,1
4,樂天桃猿,張閔勛,-1.033441,0.507117,-1.580241,-0.173547,嚴宏鈞,1.134729,-1.972057,-0.577277,-2.156622,1.761583,1.171136,1.719723,-0.638265,1.218761,1


In [4]:
monkeys = pinch_hitter_record[pinch_hitter_record["team"] == "樂天桃猿"]
dragons = pinch_hitter_record[pinch_hitter_record["team"] == "味全龍"]
hawks = pinch_hitter_record[pinch_hitter_record["team"] == "台鋼雄鷹"]
lions = pinch_hitter_record[pinch_hitter_record["team"] == "統一7-ELEVEn獅"]
guardians = pinch_hitter_record[pinch_hitter_record["team"] == "富邦悍將"]
elephants = pinch_hitter_record[pinch_hitter_record["team"] == "中信兄弟"]
print(len(monkeys), len(dragons), len(hawks), len(lions), len(guardians), len(elephants))


79 72 80 97 94 95


In [5]:
X_monkeys = monkeys[['ops_diff', 'ops_history_diff', 'ops_with_pitcherHand_diff', 'platoon_effect']]
Y_monkeys = monkeys["cluster"]
X_dragons = dragons[['ops_diff', 'ops_history_diff', 'ops_with_pitcherHand_diff', 'platoon_effect']]
Y_dragons = dragons["cluster"]
X_hawks = hawks[['ops_diff', 'ops_history_diff', 'ops_with_pitcherHand_diff', 'platoon_effect']]
Y_hawks = hawks["cluster"]
X_lions = lions[['ops_diff', 'ops_history_diff', 'ops_with_pitcherHand_diff', 'platoon_effect']]
Y_lions = lions["cluster"]
X_guardians = guardians[['ops_diff', 'ops_history_diff', 'ops_with_pitcherHand_diff', 'platoon_effect']]
Y_guardians = guardians["cluster"]
X_elephants = elephants[['ops_diff', 'ops_history_diff', 'ops_with_pitcherHand_diff', 'platoon_effect']]
Y_elephants = elephants["cluster"]
X_all = pinch_hitter_record[['ops_diff', 'ops_history_diff', 'ops_with_pitcherHand_diff', 'platoon_effect']]
Y_all = pinch_hitter_record["cluster"]



In [6]:
model = RandomForestClassifier(bootstrap=True, random_state=42)
param_dist = {
    'n_estimators': np.arange(100, 1100, 100),
    'min_samples_split': np.arange(2, 31, 1),
    'min_samples_leaf':np.arange(1, 31, 1),
    'max_depth': np.arange(2, 31, 1),
}

random_search = RandomizedSearchCV(model, param_dist, n_iter=30, cv=5)
random_search.fit(X_all, Y_all)

print(f"Best parameters: {random_search.best_params_}")
print(f"Best cross-validation R2: {round(random_search.best_score_, 4)}")


Best parameters: {'n_estimators': np.int64(500), 'min_samples_split': np.int64(14), 'min_samples_leaf': np.int64(3), 'max_depth': np.int64(20)}
Best cross-validation R2: 0.8974


In [7]:
all_model = random_search.best_estimator_
all_model

,n_estimators,np.int64(500)
,criterion,'gini'
,max_depth,np.int64(20)
,min_samples_split,np.int64(14)
,min_samples_leaf,np.int64(3)
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [8]:
# monkeys特徵重要性
all_model.fit(X_monkeys, Y_monkeys)
monkeys_feature_importance = pd.Series(all_model.feature_importances_, index=X_monkeys.columns)
monkeys_important_features = monkeys_feature_importance.sort_values(ascending=False)
# dragons特徵重要性
all_model.fit(X_dragons, Y_dragons)
dragons_feature_importance = pd.Series(all_model.feature_importances_, index=X_dragons.columns)
dragons_important_features = dragons_feature_importance.sort_values(ascending=False)
# hawks特徵重要性
all_model.fit(X_hawks, Y_hawks)
hawks_feature_importance = pd.Series(all_model.feature_importances_, index=X_hawks.columns)
hawks_important_features = hawks_feature_importance.sort_values(ascending=False)
# lions特徵重要性
all_model.fit(X_lions, Y_lions)
lions_feature_importance = pd.Series(all_model.feature_importances_, index=X_lions.columns)
lions_important_features = lions_feature_importance.sort_values(ascending=False)
# guardians特徵重要性
all_model.fit(X_guardians, Y_guardians)
guardians_feature_importance = pd.Series(all_model.feature_importances_, index=X_guardians.columns)
guardians_important_features = guardians_feature_importance.sort_values(ascending=False)
# elephants特徵重要性
all_model.fit(X_elephants, Y_elephants)
elephants_feature_importance = pd.Series(all_model.feature_importances_, index=X_elephants.columns)
elephants_important_features = elephants_feature_importance.sort_values(ascending=False)


In [19]:
importance_df = pd.DataFrame({
    "monkeys": monkeys_important_features,
    "dragons": dragons_important_features,
    "hawks": hawks_important_features,
    "lions": lions_important_features,
    "guardians": guardians_important_features,
    "elephants": elephants_important_features
})
importance_df = importance_df.sort_values(by="dragons", ascending=False)
importance_df

,monkeys,dragons,hawks,lions,guardians,elephants
ops_diff,0.239097,0.483058,0.402684,0.540325,0.317912,0.280182
ops_with_pitcherHand_diff,0.262700,0.340977,0.381296,0.237220,0.511273,0.485961
ops_history_diff,0.497193,0.173453,0.212808,0.214851,0.163707,0.206828
platoon_effect,0.001011,0.002511,0.003212,0.007604,0.007107,0.027028


In [20]:
importance_rank_df = importance_df.rank(axis=0, ascending=False, method='min')
importance_rank_df = importance_rank_df.astype(int)
importance_rank_df

,monkeys,dragons,hawks,lions,guardians,elephants
ops_diff,3,1,1,1,2,2
ops_with_pitcherHand_diff,2,2,2,2,1,1
ops_history_diff,1,3,3,3,3,3
platoon_effect,4,4,4,4,4,4


In [11]:
import shap

explainer = shap.TreeExplainer(all_model)

In [12]:
monkeys_shap_values = explainer.shap_values(X_monkeys)
monkeys_mas = np.abs(monkeys_shap_values).mean(axis=0)[:,0]
dragons_shap_values = explainer.shap_values(X_dragons)
dragons_mas = np.abs(dragons_shap_values).mean(axis=0)[:,0]
hawks_shap_values = explainer.shap_values(X_hawks)
hawks_mas = np.abs(hawks_shap_values).mean(axis=0)[:,0]
lions_shap_values = explainer.shap_values(X_lions)
lions_mas = np.abs(lions_shap_values).mean(axis=0)[:,0]
guardians_shap_values = explainer.shap_values(X_guardians)
guardians_mas = np.abs(guardians_shap_values).mean(axis=0)[:,0]
elephants_shap_values = explainer.shap_values(X_elephants)
elephants_mas = np.abs(elephants_shap_values).mean(axis=0)[:,0]

shap_df = pd.DataFrame({
    "monkeys": (monkeys_mas/monkeys_mas.sum()).round(4),
    "dragons": ((dragons_mas/dragons_mas.sum())).round(4),
    "hawks": ((hawks_mas/hawks_mas.sum())).round(4),
    "lions": ((lions_mas/lions_mas.sum())).round(4),
    "guardians": ((guardians_mas/guardians_mas.sum())).round(4),
    "elephants": ((elephants_mas/elephants_mas.sum())).round(4)
}, index=X_all.columns)

# 按照重要性排序
shap_df = shap_df.sort_values(by="monkeys", ascending=False)
shap_df

,monkeys,dragons,hawks,lions,guardians,elephants
ops_with_pitcherHand_diff,0.4609,0.4906,0.4649,0.4959,0.4950,0.4823
ops_history_diff,0.2802,0.2490,0.2461,0.2372,0.2196,0.2446
ops_diff,0.2118,0.2029,0.2384,0.2159,0.2400,0.2249
platoon_effect,0.0472,0.0575,0.0506,0.0510,0.0454,0.0482


In [13]:
# 對每個球隊內的特徵進行排名，數值越大排名越高（1為最重要的特徵）
shap_rank_df = shap_df.rank(axis=0, ascending=False, method='min')
shap_rank_df = shap_rank_df.astype(int)
shap_rank_df


,monkeys,dragons,hawks,lions,guardians,elephants
ops_with_pitcherHand_diff,1,1,1,1,1,1
ops_history_diff,2,2,2,2,3,2
ops_diff,3,3,3,3,2,3
platoon_effect,4,4,4,4,4,4


In [ ]:

from sklearn.inspection import permutation_importance
import pandas as pd


# 使用 permutation importance
monkeys_result = permutation_importance(all_model, X_monkeys, Y_monkeys, n_repeats=30, random_state=42, n_jobs=-1)
dragons_result = permutation_importance(all_model, X_dragons, Y_dragons, n_repeats=30, random_state=42, n_jobs=-1)
hawks_result = permutation_importance(all_model, X_hawks, Y_hawks, n_repeats=30, random_state=42, n_jobs=-1)
lions_result = permutation_importance(all_model, X_lions, Y_lions, n_repeats=30, random_state=42, n_jobs=-1)
guardians_result = permutation_importance(all_model, X_guardians, Y_guardians, n_repeats=30, random_state=42, n_jobs=-1)
elephants_result = permutation_importance(all_model, X_elephants, Y_elephants, n_repeats=30, random_state=42, n_jobs=-1)

# 整理結果
importance_df = pd.DataFrame({
    'feature': X_all.columns,
    'monkeys_mean_importance': monkeys_result.importances_mean,
    'monkeys_std': monkeys_result.importances_std,
    'dragons_mean_importance': dragons_result.importances_mean,
    'dragons_std': dragons_result.importances_std,
    'hawks_mean_importance': hawks_result.importances_mean,
    'hawks_std': hawks_result.importances_std,
    'lions_mean_importance': lions_result.importances_mean,
    'lions_std': lions_result.importances_std,
    'guardians_mean_importance': guardians_result.importances_mean,
    'guardians_std': guardians_result.importances_std,
    'elephants_mean_importance': elephants_result.importances_mean,
    'elephants_std': elephants_result.importances_std,
}).sort_values('elephants_mean_importance', ascending=False)

print(importance_df)


                     feature  monkeys_mean_importance  monkeys_std  \
2  ops_with_pitcherHand_diff                 0.096624     0.043048   
1           ops_history_diff                 0.089451     0.028103   
0                   ops_diff                 0.021097     0.029476   
3             platoon_effect                -0.005485     0.009631   

   dragons_mean_importance  dragons_std  hawks_mean_importance  hawks_std  \
2                 0.176389     0.036421               0.213750   0.033127   
1                 0.079630     0.026336               0.147500   0.031524   
0                 0.007407     0.019554               0.078750   0.026839   
3                 0.000000     0.000000               0.010417   0.008590   

   lions_mean_importance  lions_std  guardians_mean_importance  guardians_std  \
2               0.208935   0.042250                   0.276596       0.043949   
1               0.095876   0.022135                   0.019504       0.029338   
0               0.04